<a href="https://colab.research.google.com/github/allatop/networks/blob/main/vpn_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Как настроить VPN-сервер: что нужно для его покупки и конфигурации**

В этом руководстве разбирается:
- что такое VPN-сервер и зачем он нужен;
- что необходимо приобрести/арендовать;
- пошаговая настройка VPN на основе **WireGuard** (один из самых простых и быстрых современных протоколов);
- проверка работоспособности.

---
## 1. Что такое VPN-сервер?

**VPN (Virtual Private Network)** — виртуальная частная сеть. VPN-сервер выступает посредником между вашим устройством и интернетом:

```
Ваше устройство  ──(зашифрованный туннель)──▶  VPN-сервер  ──▶  Интернет
```

**Зачем нужен собственный VPN-сервер:**
- полный контроль над данными (нет сторонних логов);
- обход блокировок и гео-ограничений;
- безопасный доступ к домашней/офисной сети из любой точки мира;
- защита трафика в публичных Wi-Fi сетях.

---
## 2. Что нужно купить / арендовать

### 2.1 VPS (виртуальный выделенный сервер)

Основа вашего VPN — **VPS-сервер** у хостинг-провайдера.

| Параметр | Минимальные требования |
|---|---|
| ОС | Ubuntu 22.04 LTS / Debian 12 |
| CPU | 1 vCPU |
| RAM | 512 МБ |
| Диск | 10 ГБ SSD |
| Трафик | от 500 ГБ/мес |
| Стоимость | от **3–6 $/мес** |

**Популярные провайдеры:**

| Провайдер | Сайт | Особенности |
|---|---|---|
| DigitalOcean | digitalocean.com | Простой интерфейс, дата-центры по всему миру |
| Hetzner | hetzner.com | Дёшево, надёжно, серверы в Европе |
| Vultr | vultr.com | Много локаций, почасовая оплата |
| Linode / Akamai | linode.com | Стабильный, хорошая документация |
| Timeweb Cloud | timeweb.cloud | Российский провайдер, оплата в рублях |

> ⚠️ **Важно:** выбирайте локацию сервера, соответствующую вашим целям. Для обхода блокировок — сервер вне РФ (Нидерланды, Финляндия, Германия и т.д.).

### 2.2 Доменное имя (опционально)

Для удобства можно привязать домен (например, `vpn.example.com`) к IP-адресу сервера. Стоимость — от **100–150 ₽/год** у регистраторов: reg.ru, nic.ru, namecheap.com.

### 2.3 SSH-клиент (бесплатно)

Для подключения к серверу:
- **Windows:** [PuTTY](https://putty.org) или встроенный `ssh` в PowerShell/cmd
- **macOS / Linux:** встроенный `ssh` в терминале

---
## 3. Пошаговая настройка WireGuard VPN

**WireGuard** — современный быстрый протокол VPN с открытым исходным кодом. Встроен в ядро Linux начиная с версии 5.6.

### Шаг 1. Подключитесь к серверу по SSH

```bash
ssh root@<IP-адрес-сервера>
```

Замените `<IP-адрес-сервера>` на реальный IP, который выдал провайдер.

### Шаг 2. Обновите систему и установите WireGuard

```bash
apt update && apt upgrade -y
apt install -y wireguard
```

### Шаг 3. Сгенерируйте ключи сервера

```bash
# Перейдите в директорию конфигурации WireGuard
cd /etc/wireguard

# Сгенерируйте пару ключей (приватный и публичный)
wg genkey | tee server_private.key | wg pubkey > server_public.key

# Посмотрите значения ключей (они понадобятся позже)
cat server_private.key
cat server_public.key
```

### Шаг 4. Создайте конфигурационный файл сервера

```bash
nano /etc/wireguard/wg0.conf
```

Вставьте следующее содержимое, заменив `<ПРИВАТНЫЙ_КЛЮЧ_СЕРВЕРА>` и `<ВНЕШНИЙ_СЕТЕВОЙ_ИНТЕРФЕЙС>` на актуальные значения:

```ini
[Interface]
# Приватный ключ сервера (из файла server_private.key)
PrivateKey = <ПРИВАТНЫЙ_КЛЮЧ_СЕРВЕРА>

# IP-адрес сервера внутри VPN-туннеля
Address = 10.0.0.1/24

# Порт, на котором слушает WireGuard
ListenPort = 51820

# Включить IP-форвардинг и настроить NAT (замените eth0 на ваш интерфейс)
PostUp   = iptables -A FORWARD -i wg0 -j ACCEPT; iptables -t nat -A POSTROUTING -o <ВНЕШНИЙ_СЕТЕВОЙ_ИНТЕРФЕЙС> -j MASQUERADE
PostDown = iptables -D FORWARD -i wg0 -j ACCEPT; iptables -t nat -D POSTROUTING -o <ВНЕШНИЙ_СЕТЕВОЙ_ИНТЕРФЕЙС> -j MASQUERADE
```

> Узнать имя внешнего интерфейса: `ip route | grep default | awk '{print $5}'`

### Шаг 5. Включите IP-форвардинг

```bash
echo 'net.ipv4.ip_forward=1' >> /etc/sysctl.conf
sysctl -p
```

### Шаг 6. Добавьте клиента

На **сервере** выполните генерацию ключей для клиента:

```bash
cd /etc/wireguard
wg genkey | tee client_private.key | wg pubkey > client_public.key
cat client_private.key   # сохраните — понадобится для клиентского конфига
cat client_public.key    # добавляется в конфиг сервера
```

Добавьте секцию `[Peer]` в `/etc/wireguard/wg0.conf`:

```ini
[Peer]
# Публичный ключ клиента
PublicKey = <ПУБЛИЧНЫЙ_КЛЮЧ_КЛИЕНТА>

# IP-адрес клиента внутри VPN-туннеля
AllowedIPs = 10.0.0.2/32
```

### Шаг 7. Запустите WireGuard

```bash
# Запустить сервис
systemctl enable wg-quick@wg0
systemctl start  wg-quick@wg0

# Проверить статус
systemctl status wg-quick@wg0
wg show
```

### Шаг 8. Откройте порт в брандмауэре

```bash
ufw allow 51820/udp
ufw reload
```

> Если используете облачный фаервол (Security Groups у провайдера) — не забудьте также открыть UDP-порт 51820 в панели управления провайдера.

### Шаг 9. Создайте конфиг для клиентского устройства

Создайте файл `client.conf` (например, на своём компьютере) следующего содержания:

```ini
[Interface]
# Приватный ключ клиента
PrivateKey = <ПРИВАТНЫЙ_КЛЮЧ_КЛИЕНТА>

# IP-адрес клиента внутри VPN-туннеля
Address = 10.0.0.2/24

# DNS (можно использовать публичный)
DNS = 1.1.1.1

[Peer]
# Публичный ключ сервера
PublicKey = <ПУБЛИЧНЫЙ_КЛЮЧ_СЕРВЕРА>

# Внешний IP-адрес и порт сервера
Endpoint = <IP-адрес-сервера>:51820

# Весь трафик пускать через VPN
AllowedIPs = 0.0.0.0/0, ::/0

# Keepalive — поддержание соединения (актуально за NAT)
PersistentKeepalive = 25
```

Этот файл импортируется в официальное приложение **WireGuard** (доступно для Windows, macOS, Android, iOS, Linux).

---
## 4. Проверка подключения

После подключения клиента к VPN:

1. **Проверьте внешний IP** — он должен совпадать с IP вашего сервера:
   ```bash
   curl ifconfig.me
   ```

2. **На сервере** убедитесь, что клиент появился в списке:
   ```bash
   wg show
   ```
   В выводе появится строка с `latest handshake` для вашего пира.

3. **Пинг** сервера через туннель:
   ```bash
   ping 10.0.0.1
   ```

---
## 5. Краткое резюме: список покупок

| Что | Где купить | Цена |
|---|---|---|
| VPS-сервер (минимум 512 МБ RAM, 1 vCPU) | DigitalOcean / Hetzner / Vultr / Timeweb | от 3 $/мес |
| Доменное имя (опционально) | reg.ru / namecheap.com | от 100 ₽/год |
| WireGuard-клиент | wireguard.com (бесплатно) | бесплатно |

> ✅ Итого минимальный бюджет на собственный VPN-сервер — **около 300–400 ₽ в месяц** (при аренде VPS за ~3–4 $).

---
## Полезные ссылки

- [Официальная документация WireGuard](https://www.wireguard.com/quickstart/)
- [Скрипт автоматической установки WireGuard (wireguard-install)](https://github.com/Nyr/wireguard-install)
- [Сравнение VPN-протоколов: WireGuard vs OpenVPN vs IKEv2](https://www.ivpn.net/pptp-vs-ipsec-ikev2-vs-openvpn-vs-wireguard/)